In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q flask flask-cors

import os
# HTML 템플릿이 들어갈 폴더 생성
os.makedirs('templates', exist_ok=True)
print("✅ 폴더 세팅 완료!")

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>AI Music Visualizer (Real-time)</title>
    <style>
        body { margin: 0; overflow: hidden; background-color: #050510; font-family: sans-serif; }
        #ui-container { position: absolute; top: 20px; left: 20px; color: white; background: rgba(10,15,30,0.8); padding: 20px; border-radius: 10px; z-index: 100; }
        h1 { margin: 0 0 10px 0; font-size: 1.2rem; color: #a5b4fc; }
        .genre-text { font-size: 1.5rem; font-weight: bold; color: #facc15; transition: color 0.5s; text-transform: uppercase; }
        #start-btn { margin-top: 15px; padding: 10px; width: 100%; background: #dc2626; color: white; border: none; font-weight: bold; cursor: pointer; border-radius: 5px; }
        #start-btn.active { background: #16a34a; }
    </style>
</head>
<body>
    <div id="ui-container">
        <h1>AI Live Visualizer</h1>
        <div>현재 감지된 장르:</div>
        <div id="current-genre" class="genre-text">대기 중...</div>
        <button id="start-btn">마이크 켜기 / 시작</button>
    </div>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script>
        const genrePalettes = {
            'blues': { baseHue: 0.65, hueRange: 0.1, sat: 0.8, lightBase: 0.3, lightRange: 0.3 },
            'classical': { baseHue: 0.15, hueRange: 0.05, sat: 0.4, lightBase: 0.6, lightRange: 0.4 },
            'country': { baseHue: 0.08, hueRange: 0.05, sat: 0.7, lightBase: 0.4, lightRange: 0.3 },
            'disco': { baseHue: 0.85, hueRange: 0.3, sat: 1.0, lightBase: 0.5, lightRange: 0.5 },
            'hiphop': { baseHue: 0.05, hueRange: 0.1, sat: 0.9, lightBase: 0.4, lightRange: 0.4 },
            'jazz': { baseHue: 0.75, hueRange: 0.15, sat: 0.6, lightBase: 0.2, lightRange: 0.4 },
            'metal': { baseHue: 0.0, hueRange: 0.0, sat: 0.9, lightBase: 0.1, lightRange: 0.4 },
            'pop': { baseHue: 0.5, hueRange: 0.2, sat: 0.9, lightBase: 0.5, lightRange: 0.4 },
            'reggae': { baseHue: 0.3, hueRange: 0.15, sat: 0.8, lightBase: 0.4, lightRange: 0.3 },
            'rock': { baseHue: 0.6, hueRange: 0.4, sat: 0.8, lightBase: 0.3, lightRange: 0.5 }
        };

        let targetPalette = genrePalettes['blues'];
        let currentPalette = { ...targetPalette };

        let currentVolume = 0;
        let isRecording = false;

        async function startAudioLoop() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

                const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                const source = audioCtx.createMediaStreamSource(stream);
                const analyser = audioCtx.createAnalyser();
                analyser.fftSize = 256;
                source.connect(analyser);
                const dataArray = new Uint8Array(analyser.frequencyBinCount);

                function updateVolume() {
                    if(!isRecording) return;
                    analyser.getByteFrequencyData(dataArray);
                    let sum = 0;
                    for(let i = 0; i < dataArray.length; i++) sum += dataArray[i];
                    currentVolume = (sum / dataArray.length) / 30;
                    requestAnimationFrame(updateVolume);
                }
                updateVolume();

                let mediaRecorder = new MediaRecorder(stream);
                let audioChunks = [];

                mediaRecorder.ondataavailable = e => { audioChunks.push(e.data); };
                mediaRecorder.onstop = async () => {
                    if (!isRecording) return;
                    const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
                    const formData = new FormData();
                    formData.append('file', audioBlob, 'record.webm');

                    try {
                        const response = await fetch('/api/analyze', { method: 'POST', body: formData });
                        const data = await response.json();

                        if (data.genre && genrePalettes[data.genre]) {
                            targetPalette = genrePalettes[data.genre];
                            document.getElementById('current-genre').textContent = data.genre + ` (${data.probability}%)`;

                            const helper = new THREE.Color();
                            helper.setHSL(targetPalette.baseHue, 1.0, 0.6);
                            document.getElementById('current-genre').style.color = '#' + helper.getHexString();
                        }
                    } catch (err) { console.error("서버 오류", err); }

                    audioChunks = [];
                    mediaRecorder.start();
                    setTimeout(() => mediaRecorder.stop(), 3000);
                };

                mediaRecorder.start();
                setTimeout(() => mediaRecorder.stop(), 3000);
            } catch (err) {
                alert("마이크 권한이 필요합니다!");
            }
        }

        document.getElementById('start-btn').addEventListener('click', (e) => {
            isRecording = !isRecording;
            if (isRecording) {
                e.target.textContent = "분석 중...";
                e.target.className = "active";
                startAudioLoop();
            } else {
                e.target.textContent = "마이크 켜기 / 시작";
                e.target.className = "";
                currentVolume = 0;
            }
        });

        // =====================================
        // Three.js 파티클 및 씬 설정 복구 완료!
        // =====================================
        const scene = new THREE.Scene();
        scene.fog = new THREE.FogExp2(0x050510, 0.0005);
        const camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 1, 10000);
        camera.position.set(0, 500, 800);
        const renderer = new THREE.WebGLRenderer({ antialias: true });
        renderer.setSize(window.innerWidth, window.innerHeight);
        document.body.appendChild(renderer.domElement);

        const AMOUNTX = 250, AMOUNTY = 250, SEPARATION = 15;
        const numParticles = AMOUNTX * AMOUNTY;
        const geometry = new THREE.BufferGeometry();
        const positions = new Float32Array(numParticles * 3);
        const colors = new Float32Array(numParticles * 3);
        const basePositions = new Float32Array(numParticles * 3);

        let i = 0;
        for (let ix = 0; ix < AMOUNTX; ix++) {
            for (let iy = 0; iy < AMOUNTY; iy++) {
                const x = ix * SEPARATION - ((AMOUNTX * SEPARATION) / 2);
                const z = iy * SEPARATION - ((AMOUNTY * SEPARATION) / 2);

                // 위치 데이터를 꼼꼼하게 다 넣어줍니다!
                basePositions[i] = x;
                basePositions[i + 1] = 0;
                basePositions[i + 2] = z;

                positions[i] = x;
                positions[i + 1] = 0;
                positions[i + 2] = z;

                i += 3;
            }
        }
        geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
        geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));

        const material = new THREE.PointsMaterial({ size: 3, vertexColors: true, transparent: true, opacity: 0.8 });
        const particles = new THREE.Points(geometry, material);
        scene.add(particles);

        // 태양 오브젝트 부활
        const sunGeometry = new THREE.SphereGeometry(40, 32, 32);
        const sunMaterial = new THREE.MeshBasicMaterial({ color: 0xFFD700 });
        const sun = new THREE.Mesh(sunGeometry, sunMaterial);
        sun.position.set(400, 400, -300);
        scene.add(sun);

        let count = 0;
        const colorHelper = new THREE.Color();

        function lerp(start, end, amt) { return (1 - amt) * start + amt * end; }

        function animate() {
            requestAnimationFrame(animate);
            currentPalette.baseHue = lerp(currentPalette.baseHue, targetPalette.baseHue, 0.02);
            currentPalette.hueRange = lerp(currentPalette.hueRange, targetPalette.hueRange, 0.02);
            currentPalette.sat = lerp(currentPalette.sat, targetPalette.sat, 0.02);
            currentPalette.lightBase = lerp(currentPalette.lightBase, targetPalette.lightBase, 0.02);
            currentPalette.lightRange = lerp(currentPalette.lightRange, targetPalette.lightRange, 0.02);

            const posArray = particles.geometry.attributes.position.array;
            const colorArray = particles.geometry.attributes.color.array;

            let targetAmp = 1 + (currentVolume * 1.5);

            let index = 0;
            for (let ix = 0; ix < AMOUNTX; ix++) {
                for (let iy = 0; iy < AMOUNTY; iy++) {
                    const wave1 = Math.sin((ix + count * 1.2) * 0.1) * 30;
                    const wave2 = Math.cos((iy + count * 0.8) * 0.1) * 30;
                    const baseWave = wave1 + wave2 + Math.sin((ix + iy + count * 2.0) * 0.05) * 40;
                    const y = baseWave * targetAmp;

                    posArray[index + 1] = y;

                    const normalizedY = (y + 100) / 200;
                    let finalHue = (currentPalette.baseHue + (normalizedY * currentPalette.hueRange)) % 1.0;
                    if(finalHue < 0) finalHue += 1.0;

                    colorHelper.setHSL(finalHue, currentPalette.sat, currentPalette.lightBase + (normalizedY * currentPalette.lightRange));

                    colorArray[index] = colorHelper.r;
                    colorArray[index + 1] = colorHelper.g;
                    colorArray[index + 2] = colorHelper.b;
                    index += 3;
                }
            }
            particles.geometry.attributes.position.needsUpdate = true;
            particles.geometry.attributes.color.needsUpdate = true;

            // 태양 애니메이션 및 색상 동기화
            sun.position.y = 400 + Math.sin(count * 0.5) * 60;
            sun.position.x = 400 + Math.cos(count * 0.3) * 20;
            sun.position.z = -300 + Math.sin(count * 0.3) * 20;

            colorHelper.setHSL(currentPalette.baseHue, 1.0, 0.6);
            sun.material.color.lerp(colorHelper, 0.05);

            count += 0.05 + (currentVolume * 0.02);
            renderer.render(scene, camera);
        }
        animate();
    </script>
</body>
</html>

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>AI Music Visualizer (Spherical Swirl)</title>
    <style>
        body { margin: 0; overflow: hidden; background-color: #020208; font-family: sans-serif; }
        #ui-container { position: absolute; top: 20px; left: 20px; color: white; background: rgba(5, 10, 25, 0.85); padding: 20px; border-radius: 10px; z-index: 100; border: 1px solid rgba(255,255,255,0.1); }
        h1 { margin: 0 0 10px 0; font-size: 1.2rem; color: #a5b4fc; text-transform: uppercase; letter-spacing: 1px; }
        .genre-text { font-size: 1.5rem; font-weight: bold; color: #facc15; transition: color 0.5s; text-transform: uppercase; }
        #start-btn { margin-top: 15px; padding: 10px; width: 100%; background: #4f46e5; color: white; border: none; font-weight: bold; cursor: pointer; border-radius: 5px; transition: background 0.2s; }
        #start-btn.active { background: #10b981; }
    </style>
</head>
<body>
    <div id="ui-container">
        <h1>🎙️ AI Spherical Swirl</h1>
        <div>현재 감지된 장르:</div>
        <div id="current-genre" class="genre-text">대기 중...</div>
        <button id="start-btn">마이크 연결 및 시각화 가동</button>
    </div>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script>
        // 1. 더욱 비비드하고 명도가 높은 장르별 컬러 테마 정의
        const genrePalettes = {
            'blues': { baseHue: 0.62, hueRange: 0.08, sat: 1.0, lightBase: 0.4 },     // 선명한 블루 & 네온 퍼플
            'classical': { baseHue: 0.12, hueRange: 0.05, sat: 0.9, lightBase: 0.5 }, // 찬란한 화이트 골드
            'country': { baseHue: 0.07, hueRange: 0.06, sat: 1.0, lightBase: 0.45 },   // 불타는 비비드 오렌지
            'disco': { baseHue: 0.82, hueRange: 0.25, sat: 1.0, lightBase: 0.5 },    // 마젠타 & 사이안 파격 매칭
            'hiphop': { baseHue: 0.02, hueRange: 0.04, sat: 1.0, lightBase: 0.45 },   // 강렬한 싸이코패스 레드
            'jazz': { baseHue: 0.73, hueRange: 0.12, sat: 0.85, lightBase: 0.35 },    // 심야의 딥 바이올렛
            'metal': { baseHue: 0.0, hueRange: 0.02, sat: 1.0, lightBase: 0.3 },      // 다크 크림슨 레이저
            'pop': { baseHue: 0.52, hueRange: 0.15, sat: 1.0, lightBase: 0.5 },      // 청량한 아쿠아 민트
            'reggae': { baseHue: 0.28, hueRange: 0.12, sat: 1.0, lightBase: 0.45 },   // 형광 라임 그린 & 옐로우
            'rock': { baseHue: 0.58, hueRange: 0.35, sat: 1.0, lightBase: 0.4 }       // 일렉트릭 블루와 핫핑크의 대비
        };

        let targetPalette = genrePalettes['blues'];
        let currentPalette = { ...targetPalette };
        let currentVolume = 0;
        let isRecording = false;

        // 마이크 스트리밍 및 볼륨 연동
        async function startAudioLoop() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                const source = audioCtx.createMediaStreamSource(stream);
                const analyser = audioCtx.createAnalyser();
                analyser.fftSize = 256;
                source.connect(analyser);
                const dataArray = new Uint8Array(analyser.frequencyBinCount);

                function updateVolume() {
                    if(!isRecording) return;
                    analyser.getByteFrequencyData(dataArray);
                    let sum = 0;
                    for(let i = 0; i < dataArray.length; i++) sum += dataArray[i];
                    currentVolume = (sum / dataArray.length) / 80; // 감도 조절
                    requestAnimationFrame(updateVolume);
                }
                updateVolume();

                let mediaRecorder = new MediaRecorder(stream);
                let audioChunks = [];
                mediaRecorder.ondataavailable = e => { audioChunks.push(e.data); };
                mediaRecorder.onstop = async () => {
                    if (!isRecording) return;
                    const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
                    const formData = new FormData();
                    formData.append('file', audioBlob, 'record.webm');

                    try {
                        const response = await fetch('/api/analyze', { method: 'POST', body: formData });
                        const data = await response.json();
                        if (data.genre && genrePalettes[data.genre]) {
                            targetPalette = genrePalettes[data.genre];
                            document.getElementById('current-genre').textContent = data.genre + ` (${data.probability}%)`;
                        }
                    } catch (err) { console.error(err); }

                    audioChunks = [];
                    mediaRecorder.start();
                    setTimeout(() => mediaRecorder.stop(), 3000);
                };
                mediaRecorder.start();
                setTimeout(() => mediaRecorder.stop(), 3000);
            } catch (err) { alert("마이크 권한이 필요합니다!"); }
        }

        document.getElementById('start-btn').addEventListener('click', (e) => {
            isRecording = !isRecording;
            if (isRecording) {
                e.target.textContent = "🟢 실시간 오디오 동기화 중...";
                e.target.className = "active";
                startAudioLoop();
            } else {
                e.target.textContent = "마이크 연결 및 시각화 가동";
                e.target.className = "";
                currentVolume = 0;
            }
        });

        // ==========================================
        // 3D 구체 소용돌이 파티클 엔진 아키텍처
        // ==========================================
        const scene = new THREE.Scene();
        scene.fog = new THREE.FogExp2(0x020208, 0.0004);

        const camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 1, 10000);
        camera.position.set(0, 0, 900);

        const renderer = new THREE.WebGLRenderer({ antialias: true });
        renderer.setSize(window.innerWidth, window.innerHeight);
        renderer.setPixelRatio(window.devicePixelRatio);
        document.body.appendChild(renderer.domElement);

        // 고밀도 파티클 설정 (총 60,000개 입자 수학적 구체 배치)
        const AMOUNTX = 300, AMOUNTY = 200;
        const numParticles = AMOUNTX * AMOUNTY;

        const geometry = new THREE.BufferGeometry();
        const positions = new Float32Array(numParticles * 3);
        const colors = new Float32Array(numParticles * 3);
        const basePositions = new Float32Array(numParticles * 3);

        let i = 0;
        const radius = 320; // 구체 기본 반지름

        for (let ix = 0; ix < AMOUNTX; ix++) {
            for (let iy = 0; iy < AMOUNTY; iy++) {
                const u = ix / AMOUNTX;
                const v = iy / AMOUNTY;

                const theta = u * Math.PI * 2;
                const phi = Math.acos(2 * v - 1);

                // 수학적 구체 표면 좌표 산출
                const x = radius * Math.sin(phi) * Math.cos(theta);
                const y = radius * Math.cos(phi);
                const z = radius * Math.sin(phi) * Math.sin(theta);

                basePositions[i] = x;
                basePositions[i + 1] = y;
                basePositions[i + 2] = z;

                positions[i] = x;
                positions[i + 1] = y;
                positions[i + 2] = z;
                i += 3;
            }
        }
        geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
        geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));

        // ✨ 수정 포인트: 크기를 키우고(5), 겹칠수록 네온처럼 불타오르는 AdditiveBlending 탑재
        const material = new THREE.PointsMaterial({
            size: 5.5,
            vertexColors: true,
            transparent: true,
            opacity: 0.9,
            blending: THREE.AdditiveBlending, // 파티클 중첩 시 발광 효과 고조
            depthWrite: false
        });

        const particles = new THREE.Points(geometry, material);
        scene.add(particles);

        let count = 0;
        const colorHelper = new THREE.Color();
        function lerp(start, end, amt) { return (1 - amt) * start + amt * end; }

        function animate() {
            requestAnimationFrame(animate);

            // 컬러 테마 부드러운 보간
            currentPalette.baseHue = lerp(currentPalette.baseHue, targetPalette.baseHue, 0.03);
            currentPalette.sat = lerp(currentPalette.sat, targetPalette.sat, 0.03);
            currentPalette.lightBase = lerp(currentPalette.lightBase, targetPalette.lightBase, 0.03);

            const posArray = particles.geometry.attributes.position.array;
            const colorArray = particles.geometry.attributes.color.array;

            // 볼륨 감도 반응 계수
            let warpFactor = currentVolume * 1.2;

            let index = 0;
            for (let ix = 0; ix < AMOUNTX; ix++) {
                for (let iy = 0; iy < AMOUNTY; iy++) {
                    const bx = basePositions[index];
                    const by = basePositions[index + 1];
                    const bz = basePositions[index + 2];

                    // 1. 기본 자전 회전 모션 (Y축 기준 스핀)
                    const spinAngle = count * 0.2 + (iy * 0.005);
                    let x = bx * Math.cos(spinAngle) - bz * Math.sin(spinAngle);
                    let z = bx * Math.sin(spinAngle) + bz * Math.cos(spinAngle);
                    let y = by;

                    // 2. 소리 입력 시 난류(Turbulence) 소용돌이 변형 알고리즘
                    if (warpFactor > 0.05) {
                        const waveX = Math.sin(iy * 0.08 + count * 6) * 45 * warpFactor;
                        const waveY = Math.cos(ix * 0.08 + count * 6) * 45 * warpFactor;
                        const waveZ = Math.sin((ix + iy) * 0.04 + count * 6) * 45 * warpFactor;

                        x += waveX;
                        y += waveY;
                        z += waveZ;
                    }

                    posArray[index] = x;
                    posArray[index + 1] = y;
                    posArray[index + 2] = z;

                    // 3. 중심점으로부터의 거리를 계산하여 입체적인 그라데이션 컬러 부여
                    const dist = Math.sqrt(x*x + y*y + z*z);
                    const normDist = Math.min(dist / 650, 1.0);

                    let finalHue = (currentPalette.baseHue + (normDist * targetPalette.hueRange)) % 1.0;
                    if(finalHue < 0) finalHue += 1.0;

                    // 코어 영역은 하얗게 불타오르고 외곽으로 갈수록 선명한 원색 배치
                    colorHelper.setHSL(finalHue, currentPalette.sat, currentPalette.lightBase + (1.0 - normDist) * 0.4);

                    colorArray[index] = colorHelper.r;
                    colorArray[index + 1] = colorHelper.g;
                    colorArray[index + 2] = colorHelper.b;
                    index += 3;
                }
            }
            particles.geometry.attributes.position.needsUpdate = true;
            particles.geometry.attributes.color.needsUpdate = true;

            // 전체 카메라 구도 궤도 자전 회전
            particles.rotation.y = count * 0.05;
            particles.rotation.x = Math.sin(count * 0.02) * 0.2;

            count += 0.04 + (currentVolume * 0.03);
            renderer.render(scene, camera);
        }

        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });

        animate();
    </script>
</body>
</html>

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>AI Music Visualizer (Milky Way Galaxy)</title>
    <style>
        body { margin: 0; overflow: hidden; background-color: #020205; font-family: sans-serif; }
        #ui-container { position: absolute; top: 20px; left: 20px; color: white; background: rgba(4, 6, 15, 0.85); padding: 20px; border-radius: 10px; z-index: 100; border: 1px solid rgba(165, 180, 252, 0.2); box-shadow: 0 0 15px rgba(0,0,0,0.5); }
        h1 { margin: 0 0 10px 0; font-size: 1.2rem; color: #c7d2fe; text-transform: uppercase; letter-spacing: 2px; }
        .genre-text { font-size: 1.5rem; font-weight: bold; color: #facc15; transition: color 0.5s; text-transform: uppercase; letter-spacing: 1px; }
        #start-btn { margin-top: 15px; padding: 10px; width: 100%; background: #312e81; color: #e0e7ff; border: 1px solid #4338ca; font-weight: bold; cursor: pointer; border-radius: 5px; transition: all 0.2s; }
        #start-btn.active { background: #065f46; color: #a7f3d0; border-color: #047857; }
    </style>
</head>
<body>
    <div id="ui-container">
        <h1>🌌 AI Cosmic Galaxy</h1>
        <div>현재 은하 성운 장르:</div>
        <div id="current-genre" class="genre-text">대기 중...</div>
        <button id="start-btn">우주 망원경(마이크) 가동</button>
    </div>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script>
        // 장르별 은하 성운 컬러 팔레트 (신비롭고 몽환적인 우주 색감)
        const genrePalettes = {
            'blues': { baseHue: 0.64, hueRange: 0.08, sat: 1.0, lightBase: 0.35 },     // 심해 우주 네온 블루
            'classical': { baseHue: 0.13, hueRange: 0.06, sat: 0.6, lightBase: 0.55 }, // 초신성 황금빛 성단
            'country': { baseHue: 0.06, hueRange: 0.08, sat: 0.9, lightBase: 0.4 },   // 성간 먼지 오렌지 테마
            'disco': { baseHue: 0.84, hueRange: 0.20, sat: 1.0, lightBase: 0.45 },    // 웜홀 마젠타 & 사이안
            'hiphop': { baseHue: 0.01, hueRange: 0.05, sat: 1.0, lightBase: 0.4 },    // 붉은 적색왜성 광성창
            'jazz': { baseHue: 0.76, hueRange: 0.10, sat: 0.8, lightBase: 0.3 },      // 차가운 보라색 은하핵
            'metal': { baseHue: 0.0, hueRange: 0.02, sat: 1.0, lightBase: 0.25 },     // 블랙홀 주변의 불타는 기체
            'pop': { baseHue: 0.50, hueRange: 0.12, sat: 1.0, lightBase: 0.45 },      // 에메랄드빛 행성상 성운
            'reggae': { baseHue: 0.30, hueRange: 0.10, sat: 1.0, lightBase: 0.4 },    // 라임 성운 안개
            'rock': { baseHue: 0.56, hueRange: 0.30, sat: 1.0, lightBase: 0.35 }      // 코스믹 블루와 핑크의 충돌
        };

        let targetPalette = genrePalettes['blues'];
        let currentPalette = { ...targetPalette };
        let currentVolume = 0;
        let isRecording = false;

        // 마이크 스트리밍 및 최적화된 볼륨 감도 처리
        async function startAudioLoop() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                const source = audioCtx.createMediaStreamSource(stream);
                const analyser = audioCtx.createAnalyser();
                analyser.fftSize = 256;
                source.connect(analyser);
                const dataArray = new Uint8Array(analyser.frequencyBinCount);

                function updateVolume() {
                    if(!isRecording) return;
                    analyser.getByteFrequencyData(dataArray);
                    let sum = 0;
                    for(let i = 0; i < dataArray.length; i++) sum += dataArray[i];

                    // 🌟 민감도 튜닝: 나누는 수치를 75로 키워 작은 소음에는 잔잔하게 반응하도록 조정
                    currentVolume = (sum / dataArray.length) / 75;
                    requestAnimationFrame(updateVolume);
                }
                updateVolume();

                let mediaRecorder = new MediaRecorder(stream);
                let audioChunks = [];
                mediaRecorder.ondataavailable = e => { audioChunks.push(e.data); };
                mediaRecorder.onstop = async () => {
                    if (!isRecording) return;
                    const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
                    const formData = new FormData();
                    formData.append('file', audioBlob, 'record.webm');

                    try {
                        const response = await fetch('/api/analyze', { method: 'POST', body: formData });
                        const data = await response.json();
                        if (data.genre && genrePalettes[data.genre]) {
                            targetPalette = genrePalettes[data.genre];
                            document.getElementById('current-genre').textContent = data.genre + ` (${data.probability}%)`;
                        }
                    } catch (err) { console.error(err); }

                    audioChunks = [];
                    mediaRecorder.start();
                    setTimeout(() => mediaRecorder.stop(), 3000);
                };
                mediaRecorder.start();
                setTimeout(() => mediaRecorder.stop(), 3000);
            } catch (err) { alert("마이크 연결 실패"); }
        }

        document.getElementById('start-btn').addEventListener('click', (e) => {
            isRecording = !isRecording;
            if (isRecording) {
                e.target.textContent = "🚀 우주선 순항 중 (실시간 반응)";
                e.target.className = "active";
                startAudioLoop();
            } else {
                e.target.textContent = "우주 망원경(마이크) 가동";
                e.target.className = "";
                currentVolume = 0;
            }
        });

        // ==========================================
        // 🌌 나선형 우리은하 파티클 생성 시스템
        // ==========================================
        const scene = new THREE.Scene();
        scene.fog = new THREE.FogExp2(0x020205, 0.0003);

        const camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 1, 10000);
        // 은하수를 대각선 위에서 내려다보는 듯한 웅장한 시점 세팅
        camera.position.set(0, 400, 800);

        const renderer = new THREE.WebGLRenderer({ antialias: true });
        renderer.setSize(window.innerWidth, window.innerHeight);
        renderer.setPixelRatio(window.devicePixelRatio);
        document.body.appendChild(renderer.domElement);

        // 60,000개의 별(Star) 파티클 배치
        const numParticles = 60000;
        const geometry = new THREE.BufferGeometry();
        const positions = new Float32Array(numParticles * 3);
        const colors = new Float32Array(numParticles * 3);

        // 애니메이션 연산을 위한 원본 파라미터 저장소
        const starParams = [];

        const ARMS = 4; // 은하수 나선 팔 개수 (4개 주 나선 팔 구조)

        for (let i = 0; i < numParticles; i++) {
            // 거듭제곱 분포를 사용하여 은하 중심(핵)에 별이 촘촘히 뭉치도록 유도
            const r = Math.pow(Math.random(), 3.5) * 550;

            const armIndex = i % ARMS;
            const armAngle = (armIndex / ARMS) * Math.PI * 2;

            // 중심으로 갈수록 꼬이고 바깥으로 갈수록 풀어지는 나선형 각도 공식
            const spiralAngle = r * 0.008;
            const finalAngle = armAngle + spiralAngle;

            // 은하수가 자연스럽게 흩뿌려지도록 무작위 가우시안 분산 효과 추가
            const randomX = (Math.random() - 0.5) * (Math.pow(Math.random(), 1.5) * 70 + 5);
            const randomY = (Math.random() - 0.5) * (Math.pow(Math.random(), 1.5) * 40 + 5);
            const randomZ = (Math.random() - 0.5) * (Math.pow(Math.random(), 1.5) * 70 + 5);

            const x = Math.cos(finalAngle) * r + randomX;
            const y = randomY;
            const z = Math.sin(finalAngle) * r + randomZ;

            positions[i * 3] = x;
            positions[i * 3 + 1] = y;
            positions[i * 3 + 2] = z;

            starParams.push({
                radius: r,
                angle: finalAngle,
                rx: randomX,
                ry: randomY,
                rz: randomZ
            });
        }

        geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
        geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));

        // 입자 크기를 키우고(4.5), 성단이 뭉칠수록 하얗게 불타오르는 광원 블렌딩 모드 적용
        const material = new THREE.PointsMaterial({
            size: 4.5,
            vertexColors: true,
            transparent: true,
            opacity: 0.9,
            blending: THREE.AdditiveBlending,
            depthWrite: false
        });

        const particles = new THREE.Points(geometry, material);
        scene.add(particles);

        let count = 0;
        const colorHelper = new THREE.Color();
        function lerp(start, end, amt) { return (1 - amt) * start + amt * end; }

        // 우주 공간을 조작할 마우스 회전 컨트롤러 인터페이스 연결 (선택사항)
        let mouseX = 0;
        window.addEventListener('mousemove', (e) => {
            mouseX = (e.clientX - window.innerWidth / 2) * 0.2;
        });

        // ==========================================
        // 🔄 은하수 애니메이션 루프 및 사운드 매핑
        // ==========================================
        function animate() {
            requestAnimationFrame(animate);

            currentPalette.baseHue = lerp(currentPalette.baseHue, targetPalette.baseHue, 0.03);
            currentPalette.sat = lerp(currentPalette.sat, targetPalette.sat, 0.03);
            currentPalette.lightBase = lerp(currentPalette.lightBase, targetPalette.lightBase, 0.03);

            const posArray = particles.geometry.attributes.position.array;
            const colorArray = particles.geometry.attributes.color.array;

            // 소리가 들어올 때 은하수가 출렁이는 왜곡 계수 (민감도 낮춤 반영)
            let cosmicPulse = currentVolume * 1.2;

let index = 0;
            for (let i = 0; i < numParticles; i++) {
                const p = starParams[i];

                // 1. 은하 고유의 차등 회전 (중심은 빠르고 외곽은 느리게)
                const rotSpeed = count * 0.15 + (10 / (p.radius + 20));
                const animatedAngle = p.angle + rotSpeed;

                // 2. 🌟 핵심: 별들의 '밀집도'가 유기적으로 호흡하듯 흩어지고 뭉치는 효과
                let dynamicRadius = p.radius;

                if (cosmicPulse > 0.01) {
                    // (A) 확산과 수축(Breathing): 거리에 따라 물결치듯 퍼졌다가 다시 안으로 모임
                    const expansion = Math.sin(p.radius * 0.03 - count * 5) * 60 * cosmicPulse;

                    // (B) 클러스터링(Clustering): 특정 각도의 별들이 소용돌이치며 자기들끼리 뭉침
                    const cluster = Math.cos(animatedAngle * 4 + count * 3) * 40 * cosmicPulse;

                    // 본래 거리에 확산과 클러스터링 수치를 더해 밀집도를 실시간으로 왜곡
                    dynamicRadius += (expansion + cluster);
                }

                // 동적으로 변한 반지름(dynamicRadius)을 적용하여 새로운 X, Z 좌표 산출
                let x = Math.cos(animatedAngle) * dynamicRadius + p.rx;
                let z = Math.sin(animatedAngle) * dynamicRadius + p.rz;
                let y = p.ry;

                // 3. Y축(상하)의 다채로운 꽈배기(Twist) 왜곡
                if (cosmicPulse > 0.01) {
                    const wave = Math.sin(dynamicRadius * 0.04 - count * 8) * 40 * cosmicPulse;
                    const twist = Math.cos(animatedAngle * 3 + count * 5) * 30 * cosmicPulse;
                    y += (wave + twist); // 단순 상하가 아닌 꼬이면서 출렁임
                }

                posArray[index] = x;
                posArray[index + 1] = y;
                posArray[index + 2] = z;

                // 4. 컬러 스펙트럼 (기존과 동일하게 유지)
                const distFromCenter = Math.abs(dynamicRadius); // 변동된 거리를 기준으로 색상도 변함
                const normDist = Math.min(distFromCenter / 500, 1.0);

                let finalHue = (currentPalette.baseHue + (normDist * 0.15)) % 1.0;
                if(finalHue < 0) finalHue += 1.0;

                if (normDist < 0.15) {
                    colorHelper.setRGB(1.0, 0.95 - normDist, 0.9); // 은하핵 화이트
                } else {
                    colorHelper.setHSL(finalHue, currentPalette.sat, currentPalette.lightBase + (0.15 * (1.0 - normDist)));
                }

                colorArray[index] = colorHelper.r;
                colorArray[index + 1] = colorHelper.g;
                colorArray[index + 2] = colorHelper.b;
                index += 3;
            }

            particles.geometry.attributes.position.needsUpdate = true;
            particles.geometry.attributes.color.needsUpdate = true;

            // 관찰자 시점(카메라)의 드론 궤도 스무스 회전
           // 관찰자 시점(카메라)의 드론 궤도 스무스 회전
            camera.position.x = lerp(camera.position.x, mouseX, 0.05);
            camera.lookAt(0, 0, 0); // ✨ 추가된 핵심 코드: 카메라가 항상 은하 중심을 쳐다보도록 강제 고정!
            particles.rotation.y = count * 0.03; // 은하수 전체의 아주 느린 우주 자전
            // 소리가 크면 우주의 시간(애니메이션 속도)도 미세하게 가속됨
            count += 0.02 + (currentVolume * 0.01);
            renderer.render(scene, camera);
        }

        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });

        animate();
    </script>
</body>
</html>

In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>AI Music Visualizer (Curl Noise Fluid)</title>
    <style>
        body { margin: 0; overflow: hidden; background-color: #050914; font-family: 'Segoe UI', Tahoma, sans-serif; }
        #ui-container { position: absolute; top: 20px; left: 20px; color: white; background: rgba(10, 15, 30, 0.7); padding: 20px; border-radius: 12px; z-index: 100; border: 1px solid rgba(255, 255, 255, 0.1); backdrop-filter: blur(10px); }
        h1 { margin: 0 0 10px 0; font-size: 1.2rem; color: #a5b4fc; text-transform: uppercase; letter-spacing: 1px; }
        .genre-text { font-size: 1.8rem; font-weight: 800; color: #facc15; transition: color 0.5s; text-transform: uppercase; letter-spacing: 2px; }
        #start-btn { margin-top: 15px; padding: 12px; width: 100%; background: #4f46e5; color: white; border: none; font-weight: bold; cursor: pointer; border-radius: 6px; transition: background 0.3s; font-size: 1rem; }
        #start-btn:hover { background: #6366f1; }
        #start-btn.active { background: #10b981; }
    </style>
</head>
<body>
    <div id="ui-container">

        <div style="font-size: 0.9rem; color: #94a3b8; margin-bottom: 5px;">현재 감지된 장르:</div>
        <div id="current-genre" class="genre-text">대기 중...</div>
        <button id="start-btn">마이크 연결</button>
    </div>

    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>

    <script id="vertexShader" type="x-shader/x-vertex">
        uniform float uTime;
        uniform float uAudio;
        attribute vec3 customColor;
        varying vec3 vColor;

        // --- 3D Simplex Noise Function (Ian McEwan, Ashima Arts) ---
        vec3 mod289(vec3 x) { return x - floor(x * (1.0 / 289.0)) * 289.0; }
        vec4 mod289(vec4 x) { return x - floor(x * (1.0 / 289.0)) * 289.0; }
        vec4 permute(vec4 x) { return mod289(((x*34.0)+10.0)*x); }
        vec4 taylorInvSqrt(vec4 r) { return 1.79284291400159 - 0.85373472095314 * r; }

        float snoise(vec3 v) {
            const vec2  C = vec2(1.0/6.0, 1.0/3.0);
            const vec4  D = vec4(0.0, 0.5, 1.0, 2.0);
            vec3 i  = floor(v + dot(v, C.yyy));
            vec3 x0 = v - i + dot(i, C.xxx);
            vec3 g = step(x0.yzx, x0.xyz);
            vec3 l = 1.0 - g;
            vec3 i1 = min(g.xyz, l.zxy);
            vec3 i2 = max(g.xyz, l.zxy);
            vec3 x1 = x0 - i1 + C.xxx;
            vec3 x2 = x0 - i2 + C.yyy;
            vec3 x3 = x0 - D.yyy;
            i = mod289(i);
            vec4 p = permute(permute(permute(i.z + vec4(0.0, i1.z, i2.z, 1.0)) + i.y + vec4(0.0, i1.y, i2.y, 1.0)) + i.x + vec4(0.0, i1.x, i2.x, 1.0));
            float n_ = 0.142857142857;
            vec3  ns = n_ * D.wyz - D.xzx;
            vec4 j = p - 49.0 * floor(p * ns.z * ns.z);
            vec4 x_ = floor(j * ns.z);
            vec4 y_ = floor(j - 7.0 * x_);
            vec4 x = x_ *ns.x + ns.yyyy;
            vec4 y = y_ *ns.x + ns.yyyy;
            vec4 h = 1.0 - abs(x) - abs(y);
            vec4 b0 = vec4(x.xy, y.xy);
            vec4 b1 = vec4(x.zw, y.zw);
            vec4 s0 = floor(b0)*2.0 + 1.0;
            vec4 s1 = floor(b1)*2.0 + 1.0;
            vec4 sh = -step(h, vec4(0.0));
            vec4 a0 = b0.xzyw + s0.xzyw*sh.xxyy;
            vec4 a1 = b1.xzyw + s1.xzyw*sh.zzww;
            vec3 p0 = vec3(a0.xy,h.x); vec3 p1 = vec3(a0.zw,h.y);
            vec3 p2 = vec3(a1.xy,h.z); vec3 p3 = vec3(a1.zw,h.w);
            vec4 norm = taylorInvSqrt(vec4(dot(p0,p0), dot(p1,p1), dot(p2, p2), dot(p3,p3)));
            p0 *= norm.x; p1 *= norm.y; p2 *= norm.z; p3 *= norm.w;
            vec4 m = max(0.5 - vec4(dot(x0,x0), dot(x1,x1), dot(x2,x2), dot(x3,x3)), 0.0);
            m = m * m;
            return 42.0 * dot(m*m, vec4(dot(p0,x0), dot(p1,x1), dot(p2,x2), dot(p3,x3)));
        }

        vec3 snoiseVec3(vec3 x){
            float s  = snoise(vec3(x));
            float s1 = snoise(vec3(x.y - 19.1, x.z + 33.4, x.x + 47.2));
            float s2 = snoise(vec3(x.z + 74.2, x.x - 124.5, x.y + 99.4));
            return vec3(s, s1, s2);
        }

        // --- Curl Noise Function ---
        vec3 curlNoise(vec3 p) {
            const float e = 0.1;
            vec3 dx = vec3(e, 0.0, 0.0);
            vec3 dy = vec3(0.0, e, 0.0);
            vec3 dz = vec3(0.0, 0.0, e);

            vec3 p_x0 = snoiseVec3(p - dx);
            vec3 p_x1 = snoiseVec3(p + dx);
            vec3 p_y0 = snoiseVec3(p - dy);
            vec3 p_y1 = snoiseVec3(p + dy);
            vec3 p_z0 = snoiseVec3(p - dz);
            vec3 p_z1 = snoiseVec3(p + dz);

            float x = p_y1.z - p_y0.z - p_z1.y + p_z0.y;
            float y = p_z1.x - p_z0.x - p_x1.z + p_x0.z;
            float z = p_x1.y - p_x0.y - p_y1.x + p_y0.x;

            const float divisor = 1.0 / (2.0 * e);
            return normalize(vec3(x, y, z) * divisor);
        }

        void main() {
            vColor = customColor;
            vec3 pos = position;

            // 유체가 흐르는 속도와 꼬임의 크기 결정
            float noiseFreq = 0.0007;
            float timeAnim = uTime * 0.07;

            // 오디오 볼륨에 따라 소용돌이의 세기가 증폭됨
            float intensity = 30.0 + (uAudio * 350.0);

            // 3번의 루프(Octaves)를 거치며 아주 섬세하고 복잡한 유체 형태 생성
            for(int i = 0; i < 3; i++) {
                vec3 curl = curlNoise(pos * noiseFreq + timeAnim);
                pos += curl * intensity;
                noiseFreq *= 1.8; // 다음 루프에선 더 자잘한 노이즈
                intensity *= 0.45; // 대신 강도는 줄임
            }

            // 소리가 커질 때 덩어리 자체가 숨쉬듯 팽창
            pos *= 1.0 + (uAudio * 0.15);

            vec4 mvPosition = modelViewMatrix * vec4(pos, 1.0);

            // 오디오에 반응하여 입자 크기도 커짐 (역동성 추가)
            gl_PointSize = (4.0 + uAudio * 10.0) * (1000.0 / -mvPosition.z);
            gl_Position = projectionMatrix * mvPosition;
        }
    </script>

    <script id="fragmentShader" type="x-shader/x-fragment">
        varying vec3 vColor;
        void main() {
            // 원형 파티클 모양 생성 및 가장자리 부드럽게(Glow) 처리
            float dist = length(gl_PointCoord - vec2(0.5));
            if (dist > 0.5) discard;
            float alpha = pow(1.0 - (dist * 2.0), 1.5);
            gl_FragColor = vec4(vColor, alpha * 0.5);
        }
    </script>

    <script>
        // 영상 속 컬러 조합(Pink & Teal)을 비롯한 장르별 환상적인 3색 팔레트 세팅
        const genrePalettes = {
            'hiphop': ['#ff2a6d', '#05d5a1', '#ffffff'], // 비비드 핑크 & 틸 (영상과 동일한 테마)
            'blues': ['#1e3a8a', '#3b82f6', '#818cf8'],  // 딥 블루스
            'classical': ['#fef08a', '#d97706', '#ffffff'], // 우아한 골드/화이트
            'country': ['#b45309', '#d97706', '#fcd34d'], // 어스 오렌지
            'disco': ['#ec4899', '#06b6d4', '#eab308'],  // 마젠타, 사이안, 옐로우
            'jazz': ['#4c1d95', '#7c3aed', '#c4b5fd'],   // 미드나잇 퍼플
            'metal': ['#7f1d1d', '#dc2626', '#111827'],  // 블러드 레드 & 다크
            'pop': ['#f472b6', '#38bdf8', '#fbbf24'],    // 청량한 아쿠아 & 코랄
            'reggae': ['#15803d', '#eab308', '#b91c1c'], // 라스파타리안 (초/노/빨)
            'rock': ['#dc2626', '#ea580c', '#facc15']    // 플레임 (빨/주/노)
        };

        // UI 및 오디오 설정
        let currentVolume = 0;
        let isRecording = false;

        async function startAudioLoop() {
            try {
                const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
                const audioCtx = new (window.AudioContext || window.webkitAudioContext)();
                const source = audioCtx.createMediaStreamSource(stream);
                const analyser = audioCtx.createAnalyser();
                analyser.fftSize = 256;
                source.connect(analyser);
                const dataArray = new Uint8Array(analyser.frequencyBinCount);

                function updateVolume() {
                    if(!isRecording) return;
                    analyser.getByteFrequencyData(dataArray);
                    let sum = 0;
                    for(let i = 0; i < dataArray.length; i++) sum += dataArray[i];
                    // 볼륨 스무딩 및 정규화 (이 수치로 왜곡 강도가 결정됨)
                    currentVolume = (sum / dataArray.length) / 50.0;
                    requestAnimationFrame(updateVolume);
                }
                updateVolume();

                let mediaRecorder = new MediaRecorder(stream);
                let audioChunks = [];
                mediaRecorder.ondataavailable = e => { audioChunks.push(e.data); };
                mediaRecorder.onstop = async () => {
                    if (!isRecording) return;
                    const audioBlob = new Blob(audioChunks, { type: 'audio/webm' });
                    const formData = new FormData();
                    formData.append('file', audioBlob, 'record.webm');

                    try {
                        const response = await fetch('/api/analyze', { method: 'POST', body: formData });
                        const data = await response.json();
                        if (data.genre && genrePalettes[data.genre]) {
                            // AI 결과에 따라 화면 UI 색상 변경
                            document.getElementById('current-genre').textContent = data.genre + ` (${data.probability}%)`;
                            document.getElementById('current-genre').style.color = genrePalettes[data.genre][0];

                            //  유체 입자 컬러 팔레트 실시간
                            applyGenrePalette(data.genre);
                        }
                    } catch (err) { console.error(err); }

                    audioChunks = [];
                    mediaRecorder.start();
                    setTimeout(() => mediaRecorder.stop(), 3000);
                };
                mediaRecorder.start();
                setTimeout(() => mediaRecorder.stop(), 3000);
            } catch (err) { alert("마이크 연결 실패"); }
        }

        document.getElementById('start-btn').addEventListener('click', (e) => {
            isRecording = !isRecording;
            if (isRecording) {
                e.target.textContent = "...";
                e.target.className = "active";
                startAudioLoop();
                // 기본으로 힙합(영상 테마) 색상 선적용
                applyGenrePalette('hiphop');
                document.getElementById('current-genre').textContent = "오디오 분석 중...";
            } else {
                e.target.textContent = "마이크 연결";
                e.target.className = "";
                currentVolume = 0;
            }
        });

        // ==========================================
        // Three.js 씬 & GPGPU 유체 파티클 시스템 셋업
        // ==========================================
        const scene = new THREE.Scene();
        scene.fog = new THREE.FogExp2(0x050914, 0.0008);

        const camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 1, 10000);
        camera.position.set(0, 0, 800);

        const renderer = new THREE.WebGLRenderer({ antialias: true, alpha: false });
        renderer.setSize(window.innerWidth, window.innerHeight);
        renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2)); // 성능 최적화
        document.body.appendChild(renderer.domElement);

        // 파티클 개수
        const numParticles = 100000;
        const geometry = new THREE.BufferGeometry();
        const positions = new Float32Array(numParticles * 3);
        const customColors = new Float32Array(numParticles * 3);

        // 초기 입자들을 하나의 거대한 '구체(Sphere)' 형태로 빽빽하게 뭉쳐놓습니다.
        // 이 구체가 Shader를 거치면서 갈기갈기 찢어지고 소용돌이치게 됩니다.
        for (let i = 0; i < numParticles; i++) {
            let r = 250 * Math.cbrt(Math.random()); // 균일한 구체 분산
            let theta = Math.random() * 2 * Math.PI;
            let phi = Math.acos(2 * Math.random() - 1);

            positions[i * 3] = r * Math.sin(phi) * Math.cos(theta);
            positions[i * 3 + 1] = r * Math.sin(phi) * Math.sin(theta);
            positions[i * 3 + 2] = r * Math.cos(phi);
        }

        geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
        geometry.setAttribute('customColor', new THREE.BufferAttribute(customColors, 3));

        // 커스텀 ShaderMaterial 적용
        const shaderMaterial = new THREE.ShaderMaterial({
            uniforms: {
                uTime: { value: 0.0 },
                uAudio: { value: 0.0 }
            },
            vertexShader: document.getElementById('vertexShader').textContent,
            fragmentShader: document.getElementById('fragmentShader').textContent,
            transparent: true,
            blending: THREE.AdditiveBlending, // 겹칠수록 네온처럼 불타오르는 효과
            depthWrite: false
        });

        const particles = new THREE.Points(geometry, shaderMaterial);
        scene.add(particles);

        // 장르에 맞춰 색을 섞음
        function applyGenrePalette(genre) {
            const palette = genrePalettes[genre] || genrePalettes['hiphop'];
            const c1 = new THREE.Color(palette[0]);
            const c2 = new THREE.Color(palette[1]);
            const c3 = new THREE.Color(palette[2]);

            const colors = particles.geometry.attributes.customColor.array;
            for(let i = 0; i < numParticles; i++) {
                let r = Math.random();
                // 3가지 색상을 무작위 비율로 배정
                let col = r < 0.4 ? c1 : (r < 0.75 ? c2 : c3);
                colors[i*3] = col.r;
                colors[i*3+1] = col.g;
                colors[i*3+2] = col.b;
            }
            particles.geometry.attributes.customColor.needsUpdate = true;
        }

        // 초기 시작 전 색상 세팅
        applyGenrePalette('blues');

        // 마우스 카메라 이동 (살짝만 움직이도록)
        let mouseX = 0, mouseY = 0;
        window.addEventListener('mousemove', (e) => {
            mouseX = (e.clientX - window.innerWidth / 2) * 0.5;
            mouseY = (e.clientY - window.innerHeight / 2) * 0.5;
        });

        const clock = new THREE.Clock();

        function animate() {
            requestAnimationFrame(animate);

            const elapsedTime = clock.getElapsedTime();

            //셰이더로 시간과 실시간 오디오 볼륨값 전송
            shaderMaterial.uniforms.uTime.value = elapsedTime;
            // 볼륨값 보간으로 부드러운 전환
            shaderMaterial.uniforms.uAudio.value += (currentVolume - shaderMaterial.uniforms.uAudio.value) * 0.1;

            // 카메라가 소용돌이를 유려하게 바라보며 회전
            camera.position.x += (mouseX - camera.position.x) * 0.05;
            camera.position.y += (-mouseY - camera.position.y) * 0.05;
            camera.lookAt(0, 0, 0);

            // 전체 형태도 천천히 자전
            particles.rotation.y = elapsedTime * 0.1;
            particles.rotation.x = elapsedTime * 0.05;

            renderer.render(scene, camera);
        }

        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });

        animate();
    </script>
</body>
</html>

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import librosa
import numpy as np
from flask import Flask, request, jsonify, render_template
import threading
from google.colab.output import eval_js
from werkzeug.utils import secure_filename
import warnings
warnings.filterwarnings('ignore')

# 1. 모델 준비
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(model.fc.in_features, 10))

# 드라이브 경로 수정 시 이곳을 확인하세요
model_path = '/content/drive/MyDrive/AI_Visualizer/models/best_gtzan.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)
model.eval()

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

app = Flask(__name__)

# 2. 웹 페이지 띄우기
@app.route('/')
def home():
    return render_template('index.html')

# 3. 오디오 분석 API
@app.route('/api/analyze', methods=['POST'])
def analyze():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'})

    file = request.files['file']
    filepath = 'temp_audio.webm'
    file.save(filepath)

    try:
        # 오디오 로드 및 전처리
        y, sr = librosa.load(filepath, sr=22050)
        target_len = 22050 * 3 # 3초

        # 소리가 너무 짧으면 묵음으로 패딩
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))
        else:
            y = y[:target_len] # 3초만 딱 자름

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        mels_tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        # 모델 추론
        with torch.no_grad():
            outputs = model(mels_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)

        top_prob, top_idx = torch.max(probs, dim=1)
        final_genre = GENRES[top_idx.item()]

        return jsonify({
            'genre': final_genre,
            'probability': round(top_prob.item() * 100, 1)
        })

    except Exception as e:
        return jsonify({'error': str(e)})

# 4. 코랩에서 접속 가능한 공개 URL 생성 및 서버 실행
print("=====================================================")
print("🌐 아래 링크를 클릭하여 'AI 실시간 비주얼라이저'에 접속하세요!")
print(eval_js("google.colab.kernel.proxyPort(5000)"))
print("=====================================================")

# 플라스크 서버 구동
app.run(host='0.0.0.0', port=5000, debug=False)